In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2TokenizerFast
from train import Transformer

/opt/miniconda3/envs/py313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load checkpoint and tokenizer
checkpoint = torch.load('model/model.pt', map_location='cpu', weights_only=False)
tokenizer = GPT2TokenizerFast.from_pretrained('model/')
tokenizer.pad_token = tokenizer.eos_token

# Rebuild model with saved args and load weights
args = checkpoint['args']
device = 'mps' if torch.backends.mps.is_available() else 'cpu'

model = Transformer(
    vocab_size=checkpoint['vocab_size'],
    pad_token_id=checkpoint['pad_token_id'],
    d_model=args['d_model'],
    n_heads=args['n_heads'],
    n_layers=args['n_layers'],
    d_ff=args['d_ff'],
    max_seq_len=args['seq_len'],
).to(device)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f'Model loaded on {device}')
print(f"Speaker mapping: {checkpoint['speaker_to_id']}")

Model loaded on mps
Speaker mapping: {'Christian Nguyen': '<S1>', 'Daniel Ho': '<S2>', 'Fernando Ocon': '<S3>', 'Grant Qian': '<S4>', 'Jeevan Mokkala': '<S5>', 'Meta AI': '<S6>', 'Michael Hsu': '<S7>', 'Neil Pant': '<S8>'}


In [30]:
eval_sentence = """<S1>: Hey JEEVAN, tell me about machine learning <EOM>"""
tokens = tokenizer.encode(eval_sentence)
input_tensor = torch.tensor(tokens).unsqueeze(0).to(device)
logits, loss = model(input_tensor)

In [31]:
logits

tensor([[[ -0.8002,  -1.9652,  -7.2765,  ...,   1.8863,   0.8114,   1.1093],
         [ -1.5457,  -0.2589,  -8.6568,  ...,   0.8567,  -1.1370,  -1.6054],
         [  2.6826,  -1.4050, -10.6690,  ...,   1.6994,  -0.9534,   1.5873],
         ...,
         [  2.3492,   5.2120, -11.1408,  ...,   1.4503,   0.6728,  -3.4906],
         [  1.8173,  -1.5017,  -5.4794,  ...,   0.7952,  -0.0999,  19.0920],
         [  2.1302,   1.3055,  -5.8412,  ...,  15.6668,   9.8326,   1.4686]]],
       device='mps:0', grad_fn=<LinearBackward0>)

In [32]:
predicted_token = torch.argmax(logits[0, -1, :])

In [34]:
tokenizer.decode(predicted_token)

'<S7>'

In [57]:
input_sentence = "<S7> @Grant Qian tennis today? <EOM>"

def predict_next_message(input_sentence, max_len=20):
    """
    Given an input sentence, next message. Autoregressive Generation.
    Returns:
        input_text
        output_text
    """
    msg_tokens = tokenizer.encode(input_sentence)
    input_len = len(msg_tokens)
    msg_tensor = torch.tensor(msg_tokens).unsqueeze(0).to(device)
    iters = 0

    while iters < max_len:
        logits, _ = model(msg_tensor)
        next_token = torch.argmax(logits[0, -1, :]).unsqueeze(0).unsqueeze(0)
        msg_tensor = torch.cat([msg_tensor, next_token], dim=1)
        iters += 1
        if msg_tensor[0, -1].item() == 50265:
            break

    input_text = tokenizer.decode(msg_tensor[0, :input_len])
    output_text = tokenizer.decode(msg_tensor[0, input_len:])
    return input_text, output_text

input_text, output_text = predict_next_message(input_sentence)
print(f"Input:  {input_text}")
print(f"Output: {output_text}")

Input:  <S7> @Grant Qian tennis today? <EOM>
Output: <S7> @Grant Qian <EOM>
